In [1]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split

df = pd.read_csv("../data/processed/grievances_cleaned.csv")
print("Dataset shape:", df.shape)
print(df["department"].value_counts())

X = df["cleaned"]
y = df["department"]

vectorizer = TfidfVectorizer(ngram_range=(1,2), max_features=5000)
X_vec = vectorizer.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_vec, y, test_size=0.2, random_state=42, stratify=y
)
print("Train:", X_train.shape, "| Test:", X_test.shape)

Dataset shape: (300, 5)
department
Electricity     60
Sanitation      60
Water Supply    60
Transport       60
Roads           60
Name: count, dtype: int64
Train: (240, 589) | Test: (60, 589)


In [3]:
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import SVC
from sklearn.metrics import classification_report
import joblib, os

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Naive Bayes": MultinomialNB(),
    "SVM": SVC(kernel="linear", probability=True)
}

best_model, best_score, best_name = None, 0, ""

for name, model in models.items():
    model.fit(X_train, y_train)
    score = model.score(X_test, y_test)
    print(f"{name}: {score:.2%}")
    if score > best_score:
        best_score, best_model, best_name = score, model, name

print(f"\nBest: {best_name} ({best_score:.2%})")
print(classification_report(y_test, best_model.predict(X_test)))

Logistic Regression: 100.00%
Naive Bayes: 100.00%
SVM: 100.00%

Best: Logistic Regression (100.00%)
              precision    recall  f1-score   support

 Electricity       1.00      1.00      1.00        12
       Roads       1.00      1.00      1.00        12
  Sanitation       1.00      1.00      1.00        12
   Transport       1.00      1.00      1.00        12
Water Supply       1.00      1.00      1.00        12

    accuracy                           1.00        60
   macro avg       1.00      1.00      1.00        60
weighted avg       1.00      1.00      1.00        60



In [5]:
os.makedirs("../models", exist_ok=True)
joblib.dump(best_model, "../models/department_classifier.pkl")
joblib.dump(vectorizer, "../models/tfidf_vectorizer.pkl")
print("Models saved!")

# Quick test
test = ["Water supply cut off for 3 days urgent help"]
cleaned_test = vectorizer.transform(test)
print("Test prediction:", best_model.predict(cleaned_test)[0])

Models saved!
Test prediction: Water Supply
